In [1]:
import os
import sys
import re

def check_train_folder(train_folder):
    # Get all scenario folders in the train folder.
    scenarios = [d for d in os.listdir(train_folder) if os.path.isdir(os.path.join(train_folder, d))]
    required_directions = {'0', '1', '2', '3', '4', '5'}

    # Overall report for missing radar files.
    overall_missing_data = {}  # { scenario: { agent: { timestamp: [missing_directions] } } }
    # Overall agent timestamps per scenario.
    overall_agent_timestamps = {}  # { scenario: { agent: [timestamp, ...] } }

    # Regex to match files: <timestamp>_radar_<direction>.pcd
    pattern = re.compile(r'^(?P<timestamp>\d+)_radar(?P<direction>0|1|2|3|4|5)\.npy$')

    for scenario in scenarios:
        scenario_path = os.path.join(train_folder, scenario)
        agents = [d for d in os.listdir(scenario_path) if os.path.isdir(os.path.join(scenario_path, d))]
        scenario_missing = {}          # missing radar info for the scenario
        agent_timestamp_dict = {}      # {agent: {timestamp: set(directions)} }
        scenario_agent_timestamps = {} # {agent: set(timestamps)}
        global_timestamps = set()      # union of all timestamps in this scenario

        # Process each agent in the scenario.
        for agent in agents:
            agent_path = os.path.join(scenario_path, agent)
            agent_timestamp_dict[agent] = {}
            scenario_agent_timestamps[agent] = set()
            for file in os.listdir(agent_path):
                match = pattern.match(file)
                if match:
                    timestamp = match.group("timestamp")
                    direction = match.group("direction")
                    scenario_agent_timestamps[agent].add(timestamp)
                    if timestamp not in agent_timestamp_dict[agent]:
                        agent_timestamp_dict[agent][timestamp] = set()
                    agent_timestamp_dict[agent][timestamp].add(direction)
                    global_timestamps.add(timestamp)

        # Save each agent's found timestamps for reporting.
        overall_agent_timestamps[scenario] = {agent: sorted(list(ts_set)) for agent, ts_set in scenario_agent_timestamps.items()}

        # For every agent, check every timestamp (from the union) for missing radar files.
        for agent, ts_dict in agent_timestamp_dict.items():
            for ts in global_timestamps:
                if ts in ts_dict:
                    missing_dirs = required_directions - ts_dict[ts]
                    if missing_dirs:
                        if agent not in scenario_missing:
                            scenario_missing[agent] = {}
                        scenario_missing[agent][ts] = sorted(missing_dirs)
                else:
                    # Agent does not have any file for this timestamp.
                    if agent not in scenario_missing:
                        scenario_missing[agent] = {}
                    scenario_missing[agent][ts] = sorted(required_directions)

        # Detailed report for the scenario.
        print(f"Scenario: {scenario}")
        for agent in agents:
            print(f"  Agent: {agent}")
            timestamps_found = sorted(list(scenario_agent_timestamps.get(agent, set())))
            print(f"    Timestamps found: {', '.join(timestamps_found) if timestamps_found else 'None'}")
            if agent in scenario_missing:
                for ts, missing_dirs in sorted(scenario_missing[agent].items()):
                    print(f"    Timestamp {ts}: missing {', '.join(missing_dirs)}")
            else:
                print("    All timestamps have complete radar files.")

        # Check for consistency of timestamps among agents.
        unique_timestamp_sets = set(frozenset(ts) for ts in scenario_agent_timestamps.values())
        if len(unique_timestamp_sets) == 1:
            print("    All agents have the same timestamps.")
        else:
            print("    Inconsistent timestamps among agents:")
            for agent, ts_set in scenario_agent_timestamps.items():
                print(f"      {agent}: {sorted(ts_set)}")
        print("-" * 50)

        if scenario_missing:
            overall_missing_data[scenario] = scenario_missing

    # Summary report of missing radar files.
    if overall_missing_data:
        print("\nSummary of Missing Radar Files:")
        for scenario, agents_data in overall_missing_data.items():
            for agent, timestamps in agents_data.items():
                for ts, missing_dirs in sorted(timestamps.items()):
                    print(f"Scenario: {scenario}, Agent: {agent}, Timestamp: {ts} -> missing {', '.join(missing_dirs)}")
    else:
        print("All scenarios and agents have complete radar files for each timestamp.")

In [2]:
 check_train_folder("/home/ws-ids-es3-01/Developer/Dataset/Dataset_OPV2V/validate_additional")

Scenario: 2021_08_21_17_30_41
  Agent: 2506
    Timestamps found: 000069, 000071, 000073, 000075, 000077, 000079, 000081, 000083, 000085, 000087, 000089, 000091, 000093, 000095, 000097, 000099, 000101, 000103, 000105, 000107, 000109, 000111, 000113, 000115, 000117, 000119, 000121, 000123, 000125, 000127, 000129, 000131, 000133, 000135, 000137, 000139, 000141, 000143, 000145, 000147, 000149, 000151, 000153, 000155, 000157, 000159, 000161, 000163, 000165, 000167, 000169, 000171, 000173, 000175, 000177, 000179, 000181, 000183, 000185, 000187, 000189, 000191, 000193, 000195, 000197, 000199, 000201, 000203, 000205, 000207, 000209, 000211, 000213, 000215, 000217, 000219, 000221, 000223, 000225, 000227, 000229, 000231, 000233, 000235, 000237, 000239, 000241, 000243, 000245, 000247, 000249, 000251, 000253, 000255, 000257, 000259, 000261, 000263, 000265, 000267, 000269, 000271, 000273, 000275, 000277, 000279, 000281, 000283, 000285, 000287, 000289, 000291, 000293, 000295, 000297, 000299, 000301